In [ ]:
from pathlib import Path
import re
import pandas as pd
from IPython.display import Markdown, display

# The notebook is expected to run from the repository root.
DATA_DIR = Path.cwd() / 'Data'
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / 'Data'

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 80)
print(f'Input directory: {DATA_DIR.resolve()}')

dfs = {
    'patients': pd.read_csv(DATA_DIR / 'patients.csv'),
    'encounters': pd.read_parquet(DATA_DIR / 'encounters.parquet'),
    'symptoms': pd.read_csv(DATA_DIR / 'symptoms.csv'),
    'medications': pd.read_csv(DATA_DIR / 'medications.csv'),
    'conditions': pd.read_excel(DATA_DIR / 'conditions.xlsx'),
}


# EDA

In [ ]:
def exploratory_data_analysis(dataframe):
    print('*'*40+'Data Shape'+'*'*40)
    print(f'Rows: {len(dataframe):,}')
    print(f'Columns: {dataframe.shape[1]}')

    print("\n\n"+ '*'*40+'Data Preview'+'*'*40)
    display(dataframe.sample(min(10, len(dataframe))))

    print("\n\n"+ '*'*40+'Data types'+'*'*40)
    display(dataframe.dtypes.rename('dtype').to_frame())

    print("\n\n"+ '*'*40+'Missing values'+'*'*40)
    missing = dataframe.isna().sum().rename('missing_count').to_frame()
    missing['missing_percent'] = (missing['missing_count'] / len(dataframe) * 100).round(2)
    display(missing.sort_values('missing_count', ascending=False))

    print("\n\n"+ '*'*40+'Data quality checks'+'*'*40)
    print(f'Duplicate rows: {dataframe.duplicated().sum():,}')
    for column in dataframe.columns:
        duplicate_count = dataframe[column].duplicated().sum()
        if dataframe[column].nunique(dropna=True) == len(dataframe) and duplicate_count == 0:
            print(f'Potential unique identifier: {column}')

    categorical_columns = dataframe.select_dtypes(include=['object', 'string', 'category', 'bool']).columns
    for column in categorical_columns:
        if dataframe[column].nunique(dropna=True) <= 20:
            print("\n"+ '*'*20+f'{column} value counts'+'*'*20)
            display(dataframe[column].value_counts(dropna=False).rename('count').to_frame())

    date_columns = [column for column in dataframe.columns if any(
        marker in column.upper() for marker in ('DATE', 'START', 'STOP')
    )]
    print("\n\n"+ '*'*40+'Date ranges'+'*'*40)
    for column in date_columns:
        parsed = pd.to_datetime(dataframe[column], errors='coerce', utc=True)
        if parsed.notna().any():
            print(f'{column}: {parsed.min()} to {parsed.max()}')

    numeric_columns = dataframe.select_dtypes(include='number').columns
    if len(numeric_columns) > 0:
        print("\n\n"+ '*'*40+'Numeric summary'+'*'*40)
        display(dataframe[numeric_columns].describe().T)


# Keep explicit table names for the relationship checks below.
patients, encounters, symptoms, medications, conditions = (
    dfs[name] for name in ('patients', 'encounters', 'symptoms', 'medications', 'conditions')
)

for table_name, dataframe in dfs.items():
    print(f'\n\n\n{table_name.upper()}')
    exploratory_data_analysis(dataframe)
    print("*"*100)
    print("*"*100)

# 2. Summary of issues and recommendations

| Table | Area | Main issue | Recommendation |
|---|---|---|---|
| `patients` | Missingness | `GENDER` and `DEATHDATE` are completely null | Drop the cols or Preserve them for future use make sure to use NULL for current values. |
| `patients` | Privacy | Direct identifiers such as SSN, passport, and driver license are present | Restrict access and remove unnecessary identifiers from analytical marts. |
| `patients` | Schema | categorical fields require validation | normalize categorical values. |
| `encounters` | Schema | Source columns use inconsistent names | Rename `Id` to `encounter_id` and `PATIENT` to `patient_id` |
| `encounters` | Dates | Timestamps are stored as strings and include UTC markers | convert `START` and `STOP` pandas date format. |
| `symptoms` | Structure | `GENDER`, `RACE`, `ETHNICITY` fields are maybe repative of patient table|   |
| `symptoms` | Missingness | `GENDER` and `AGE_END` are completely null | Preserve and flag these missing fields. |
| `symptoms` | Validation | `NUM_SYMPTOMS` may not match parseable values | Recalculate counts from successfully parsed symptom pairs. |
| `medications` | Schema | Source columns use inconsistent names and no explicit medication key exists | Standardize names and build a deterministic medication record key. |
| `medications` | Dates | Medication timestamps are stored as strings |  |
| `medications` | Validation | Medication costs require validation | Convert costs to numeric and investigate negative values. |
| `conditions` | Schema | convert `START` and `STOP` pandas date format. `STOP`  is all null |  |
| `conditions` | Schema | case mismatch in  `CODE` and `DESCRIPTION` pandas date format. |  use consistance case for `DESCRIPTION` |

# 3. Cardinality Check

In [ ]:
dataframes = dfs

# Column cardinality: distinct values and distinctness percentage.
cardinality_rows = []
for table_name, dataframe in dataframes.items():
    for column in dataframe.columns:
        non_null_count = dataframe[column].notna().sum()
        distinct_count = dataframe[column].nunique(dropna=True)
        cardinality_rows.append({
            'table': table_name,
            'column': column,
            'rows': len(dataframe),
            'distinct_values': distinct_count,
            'distinct_percent': round(distinct_count / non_null_count * 100, 2) if non_null_count else 0,
            'null_count': dataframe[column].isna().sum(),
        })

cardinality = (
    pd.DataFrame(cardinality_rows)
    .sort_values(['table', 'distinct_percent'], ascending=[True, False])
    .reset_index(drop=True)
)
display(cardinality)

# Relationship cardinality for patient-linked tables.
patient_cardinality = []
for table_name, dataframe in dataframes.items():
    patient_column = next(
        (column for column in dataframe.columns if column.upper() in {'PATIENT', 'PATIENT_ID'}),
        None,
    )
    if patient_column is not None:
        unique_patients = dataframe[patient_column].nunique(dropna=True)
        patient_cardinality.append({
            'table': table_name,
            'patient_column': patient_column,
            'rows': len(dataframe),
            'unique_patients': unique_patients,
            'rows_per_patient': round(len(dataframe) / unique_patients, 2) if unique_patients else 0,
        })

display(pd.DataFrame(patient_cardinality))

# Referential integrity: verify that every child-table patient exists in patients.
master_patient_ids = set(
    patients['PATIENT_ID'].dropna().astype(str).str.strip().str.upper()
)
integrity_rows = []
for table_name, dataframe in dataframes.items():
    patient_column = next(
        (column for column in dataframe.columns if column.upper() in {'PATIENT', 'PATIENT_ID'}),
        None,
    )
    if patient_column is not None:
        table_patient_ids = set(
            dataframe[patient_column].dropna().astype(str).str.strip().str.upper()
        )
        missing_patient_ids = table_patient_ids - master_patient_ids
        integrity_rows.append({
            'table': table_name,
            'patient_column': patient_column,
            'unique_patients': len(table_patient_ids),
            'matched_patients': len(table_patient_ids - missing_patient_ids),
            'missing_patient_count': len(missing_patient_ids),
            'all_patients_in_master': len(missing_patient_ids) == 0,
            'sample_missing_ids': sorted(missing_patient_ids)[:10],
        })

integrity = pd.DataFrame(integrity_rows)
display(integrity)

In [ ]:
# Referential integrity: verify that every referenced encounter exists in encounters.
encounter_id_column = next(
    column for column in encounters.columns if column.upper() in {'ID', 'ENCOUNTER_ID'}
)
master_encounter_ids = set(
    encounters[encounter_id_column].dropna().astype(str).str.strip().str.upper()
)
encounter_integrity_rows = []

for table_name in ['medications', 'conditions']:
    dataframe = dataframes[table_name]
    encounter_column = next(
        column for column in dataframe.columns if column.upper() in {'ENCOUNTER', 'ENCOUNTER_ID'}
    )
    encounter_ids = set(
        dataframe[encounter_column].dropna().astype(str).str.strip().str.upper()
    )
    missing_encounter_ids = encounter_ids - master_encounter_ids
    encounter_integrity_rows.append({
        'table': table_name,
        'encounter_column': encounter_column,
        'unique_encounters': len(encounter_ids),
        'matched_encounters': len(encounter_ids - missing_encounter_ids),
        'missing_encounter_count': len(missing_encounter_ids),
        'all_encounters_in_master': len(missing_encounter_ids) == 0,
        'sample_missing_ids': sorted(missing_encounter_ids)[:10],
    })

encounter_integrity = pd.DataFrame(encounter_integrity_rows)
display(encounter_integrity)